# ECA-0.4 — Adaptive Recovery and T4 Scaling

This notebook trains the 256-hidden / 2-depth configuration with validation-gated best checkpoints. If validation stalls or regresses, the trainer restores the best checkpoint, lowers the learning rate, and adds a bounded recovery budget. Select a T4 GPU and run from top to bottom.

In [ ]:
import os, torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
!nvidia-smi

In [ ]:
REPO='/content/eca'
BRANCH='eca-0.4-adaptive-recovery'
if not os.path.isdir(REPO):
    !git clone --branch $BRANCH https://github.com/swasthik-s/eca.git $REPO
%cd /content/eca
!git fetch origin
!git checkout $BRANCH
!git pull --ff-only origin $BRANCH
!pip -q install -e '.[train,dev]'
print('Branch:', BRANCH)

In [ ]:
!pytest -q tests/test_parallel_matrix.py
print('Existing ECA-0.3 architecture tests passed.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE='/content/drive/MyDrive/ECA/checkpoints'
os.makedirs(BASE, exist_ok=True)
print('Checkpoint directory:', BASE)

## Configure the scale experiment

Start with 256 hidden units and depth 2. If stable, repeat with 384/3 and 512/4, using different checkpoint filenames. Do not run several scales concurrently.

In [ ]:
DATASET_ID='HuggingFaceFW/fineweb-edu'
DATASET_CONFIG='sample-10BT'
DOCUMENTS=2000
MAX_BYTES=2_000_000
STEPS=600
SEQUENCE_LENGTH=256
BATCH_SIZE=8
HIDDEN_SIZE=256
EMBEDDING_SIZE=64
DEPTH=2
KERNEL_SIZE=5
SEED=17
EVAL_EVERY=50
PATIENCE=3
RECOVERY_EXTRA_STEPS=200
MAX_RECOVERIES=3
RUN_NAME=f'eca_0_4_h{HIDDEN_SIZE}_d{DEPTH}'
BEST=f'{BASE}/{RUN_NAME}_best.pt'
LATEST=f'{BASE}/{RUN_NAME}_latest.pt'
print('Run:', RUN_NAME, '| initial steps:', STEPS, '| max extra steps:', RECOVERY_EXTRA_STEPS*MAX_RECOVERIES)

## Train with validation checks and automatic recovery

The controller chooses checkpoints using held-out validation bits-per-byte (BPB), not training loss. A recovery restores model, optimizer and scaler, reduces the learning rate, resets temporary recurrent activations, and adds extra steps. Recovery is capped at three attempts.

In [ ]:
!python -m eca.training.train_adaptive \\
  --dataset hf \\
  --dataset-id $DATASET_ID \\
  --dataset-config $DATASET_CONFIG \\
  --documents $DOCUMENTS \\
  --max-bytes $MAX_BYTES \\
  --steps $STEPS \\
  --sequence-length $SEQUENCE_LENGTH \\
  --batch-size $BATCH_SIZE \\
  --hidden-size $HIDDEN_SIZE \\
  --embedding-size $EMBEDDING_SIZE \\
  --depth $DEPTH \\
  --kernel-size $KERNEL_SIZE \\
  --seed $SEED \\
  --eval-every $EVAL_EVERY \\
  --patience $PATIENCE \\
  --recovery-extra-steps $RECOVERY_EXTRA_STEPS \\
  --max-recoveries $MAX_RECOVERIES \\
  --output "$LATEST" \\
  --best-checkpoint "$BEST"

In [ ]:
import os, torch
assert os.path.isfile(BEST), f'Best checkpoint missing: {BEST}'
best=torch.load(BEST, map_location='cpu', weights_only=True)
print('Architecture:', best['architecture'])
print('Best step:', best['step'])
print('Best validation BPB:', best['validation_bpb'])
print('Recovery count:', best['recovery_count'])
print('Model config:', best['config'])
print('Best checkpoint MB:', round(os.path.getsize(BEST)/(1024**2),2))
print('Latest checkpoint exists:', os.path.isfile(LATEST))
!nvidia-smi

## How to interpret the result

- A new best checkpoint means held-out BPB improved by at least the configured minimum.
- A recovery means the validation metric stalled; the trainer restored the best state and allocated additional steps at a lower learning rate.
- Repeated recovery without improvement is a signal to stop and diagnose data, optimization or model capacity—not to keep training indefinitely.
- BPB only measures next-byte prediction. It does not establish reasoning, semantic understanding or long-range recall.

For scaling, change `HIDDEN_SIZE`, `DEPTH`, `EMBEDDING_SIZE`, and `RUN_NAME` in the configuration cell, then rerun the training and verification cells. Suggested next configurations: 384/3/64 and 512/4/96. Use a separate checkpoint name for every run.